In [1]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import FormulaRule, ColorScaleRule
from openpyxl.chart import BarChart, Reference


def generate_apsara_report(input_csv, output_xlsx, date_label, report_label="Weekly"):
    df = pd.read_csv(input_csv)

    PIVOT_COLS = [
        "Restaurant", "Gross sale", "Net Sales Total", "Offline Sales", "Online Sales",
        "Gross Sale (Zomato)", "Gross Sale (Swiggy)", "Total Discount",
        "No Of Orders", "No Of Orders (Offline)", "No Of Orders (Online)",
        "No Of Orders (Zomato)", "No Of Orders (Swiggy)",
        "Avg Bill Value", "Avg Bill Value (Zomato)", "Avg Bill Value (Swiggy)",
    ]
    pivot_df = df[PIVOT_COLS].sort_values("Net Sales Total", ascending=False).reset_index(drop=True)

    TB_COLS = ["Restaurant", "Net Sales Total", "No Of Orders", "Avg Bill Value"]
    ranked = df[TB_COLS].sort_values("Net Sales Total", ascending=False).reset_index(drop=True)
    top10 = ranked.head(10).reset_index(drop=True)
    bottom10 = ranked.tail(10).sort_values("Net Sales Total", ascending=True).reset_index(drop=True)
    top10.insert(0, "Rank", range(1, len(top10) + 1))
    bottom10.insert(0, "Rank", range(1, len(bottom10) + 1))

    HEADER_BLUE, HEADER_ORANGE = "FF1F4E78", "FFC55A11"
    WHITE_BOLD = Font(bold=True, color="FFFFFFFF", size=10)
    TITLE_FONT = Font(bold=True, color="FFFFFFFF", size=16)
    SUBTITLE_FONT = Font(bold=False, color="FFFFFFFF", size=10)
    BLUE_LABEL_FONT = Font(bold=True, color="FF1F4E78", size=12)
    RED_LABEL_FONT = Font(bold=True, color="FFC00000", size=12)
    ORANGE_COLS = {"Gross Sale (Zomato)", "Gross Sale (Swiggy)", "No Of Orders (Zomato)",
                   "No Of Orders (Swiggy)", "Avg Bill Value (Zomato)", "Avg Bill Value (Swiggy)"}

    def fill(color):
        return PatternFill(start_color=color, end_color=color, fill_type="solid")

    def write_header_row(ws, row, headers, orange_cols=None):
        orange_cols = orange_cols or set()
        for i, h in enumerate(headers, start=1):
            cell = ws.cell(row=row, column=i, value=h)
            cell.font = WHITE_BOLD
            cell.fill = fill(HEADER_ORANGE if h in orange_cols else HEADER_BLUE)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    def write_title_block(ws, title, subtitle, n_cols):
        ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
        c1 = ws.cell(row=1, column=1, value=title)
        c1.font = TITLE_FONT; c1.fill = fill(HEADER_BLUE)
        c1.alignment = Alignment(horizontal="left", vertical="center")
        ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=n_cols)
        c2 = ws.cell(row=2, column=1, value=subtitle)
        c2.font = SUBTITLE_FONT; c2.fill = fill(HEADER_BLUE)
        c2.alignment = Alignment(horizontal="left", vertical="center")

    wb = Workbook()

    # ---- Sheet 1: Restaurant Pivot ----
    ws1 = wb.active
    ws1.title = "Restaurant Pivot"
    write_title_block(ws1,
        "APSARA SALES REPORT — RESTAURANT-WISE PERFORMANCE PIVOT",
        f"Report Date: {date_label} ({report_label})  |  Sorted by Net Sales Total (High to Low)  |  "
        "Orange columns = Online (Zomato vs Swiggy) comparison  |  Full raw data in 'Raw Data' tab",
        n_cols=len(PIVOT_COLS))

    HEADER_ROW = 4
    write_header_row(ws1, HEADER_ROW, PIVOT_COLS, ORANGE_COLS)

    money_cols = {"Gross sale", "Net Sales Total", "Offline Sales", "Online Sales",
                  "Gross Sale (Zomato)", "Gross Sale (Swiggy)", "Total Discount"}
    count_cols = {"No Of Orders", "No Of Orders (Offline)", "No Of Orders (Online)",
                  "No Of Orders (Zomato)", "No Of Orders (Swiggy)"}
    avg_cols = {"Avg Bill Value", "Avg Bill Value (Zomato)", "Avg Bill Value (Swiggy)"}

    for r_idx, row in enumerate(pivot_df.itertuples(index=False), start=HEADER_ROW + 1):
        for c_idx, col_name in enumerate(PIVOT_COLS, start=1):
            cell = ws1.cell(row=r_idx, column=c_idx, value=row[c_idx - 1])
            if col_name in money_cols or col_name in count_cols:
                cell.number_format = "#,##0"
            elif col_name in avg_cols:
                cell.number_format = "#,##0.00"

    total_row = HEADER_ROW + 1 + len(pivot_df)
    ws1.cell(row=total_row, column=1, value="GRAND TOTAL").font = Font(bold=True)
    for c_idx, col_name in enumerate(PIVOT_COLS[1:], start=2):
        if col_name in avg_cols:
            continue
        col_letter = get_column_letter(c_idx)
        f = ws1.cell(row=total_row, column=c_idx,
                      value=f"=SUM({col_letter}{HEADER_ROW+1}:{col_letter}{total_row-1})")
        f.font = Font(bold=True); f.number_format = "#,##0"

    ws1.freeze_panes = "B5"
    ws1.column_dimensions["A"].width = 34
    ws1.column_dimensions["B"].width = 15
    for i in range(3, len(PIVOT_COLS) + 1):
        ws1.column_dimensions[get_column_letter(i)].width = 16

    DATA_START, DATA_END = HEADER_ROW + 1, total_row - 1

    # ---- Net Sales Total: green shade, dark (high) to light (low) ----
    net_sales_letter = get_column_letter(PIVOT_COLS.index("Net Sales Total") + 1)
    net_sales_range = f"{net_sales_letter}{DATA_START}:{net_sales_letter}{DATA_END}"
    ws1.conditional_formatting.add(net_sales_range, ColorScaleRule(
        start_type="min", start_color="E2F0D9",
        end_type="max", end_color="375623"))

    # ---- Zomato vs Swiggy per-row comparison (dark = higher of the two, light = lower) ----
    zomato_letter = get_column_letter(PIVOT_COLS.index("Gross Sale (Zomato)") + 1)
    swiggy_letter = get_column_letter(PIVOT_COLS.index("Gross Sale (Swiggy)") + 1)
    zomato_range = f"{zomato_letter}{DATA_START}:{zomato_letter}{DATA_END}"
    swiggy_range = f"{swiggy_letter}{DATA_START}:{swiggy_letter}{DATA_END}"
    dark_red, light_red = fill("C00000"), fill("FCE4E4")
    dark_teal, light_teal = fill("1F6F5C"), fill("E2F1EE")
    white_font = Font(size=10, color="FFFFFF")

    ws1.conditional_formatting.add(zomato_range, FormulaRule(
        formula=[f"{zomato_letter}{DATA_START}>{swiggy_letter}{DATA_START}"], fill=dark_red, font=white_font))
    ws1.conditional_formatting.add(zomato_range, FormulaRule(
        formula=[f"{zomato_letter}{DATA_START}<={swiggy_letter}{DATA_START}"], fill=light_red))
    ws1.conditional_formatting.add(swiggy_range, FormulaRule(
        formula=[f"{swiggy_letter}{DATA_START}>{zomato_letter}{DATA_START}"], fill=dark_teal, font=white_font))
    ws1.conditional_formatting.add(swiggy_range, FormulaRule(
        formula=[f"{swiggy_letter}{DATA_START}<={zomato_letter}{DATA_START}"], fill=light_teal))

    # ---- Sheet 2: Top10 & Bottom10 ----
    ws2 = wb.create_sheet("Top10 & Bottom10 Outlets")
    tb_headers = ["Rank", "Restaurant", "Net Sales Total", "No Of Orders", "Avg Bill Value"]
    write_title_block(ws2, f"TOP 10 & BOTTOM 10 PERFORMING OUTLETS ({report_label.upper()}) — {date_label}",
                       "Ranked by Net Sales Total (₹)", n_cols=len(tb_headers))
    ws2.cell(row=4, column=1, value="TOP 10 OUTLETS").font = BLUE_LABEL_FONT
    write_header_row(ws2, 5, tb_headers)
    for i, row in enumerate(top10.itertuples(index=False), start=6):
        for c_idx, val in enumerate(row, start=1):
            cell = ws2.cell(row=i, column=c_idx, value=val)
            if c_idx == 3: cell.number_format = "#,##0"
            elif c_idx == 5: cell.number_format = "#,##0.00"

    top_data_start, top_data_end = 6, 6 + len(top10) - 1
    bottom_label_row = 6 + len(top10) + 1
    ws2.cell(row=bottom_label_row, column=1, value="BOTTOM 10 OUTLETS").font = RED_LABEL_FONT
    write_header_row(ws2, bottom_label_row + 1, tb_headers)
    for i, row in enumerate(bottom10.itertuples(index=False), start=bottom_label_row + 2):
        for c_idx, val in enumerate(row, start=1):
            cell = ws2.cell(row=i, column=c_idx, value=val)
            if c_idx == 3: cell.number_format = "#,##0"
            elif c_idx == 5: cell.number_format = "#,##0.00"

    bottom_data_start = bottom_label_row + 2
    bottom_data_end = bottom_data_start + len(bottom10) - 1

    ws2.column_dimensions["A"].width = 8
    ws2.column_dimensions["B"].width = 34
    ws2.column_dimensions["C"].width = 16
    ws2.column_dimensions["D"].width = 14
    ws2.column_dimensions["E"].width = 16

    # ---- Charts: Top10 & Bottom10 ----
    chart_top = BarChart()
    chart_top.type = "col"
    chart_top.title = "Top 10 Outlets — Net Sales Total (₹)"
    chart_top.style = 10
    chart_top.y_axis.title = "Net Sales Total"
    chart_top.x_axis.title = "Restaurant"
    data_ref = Reference(ws2, min_col=3, min_row=5, max_row=top_data_end)
    cats_ref = Reference(ws2, min_col=2, min_row=top_data_start, max_row=top_data_end)
    chart_top.add_data(data_ref, titles_from_data=True)
    chart_top.set_categories(cats_ref)
    chart_top.width, chart_top.height = 20, 10
    ws2.add_chart(chart_top, "G4")

    chart_bottom = BarChart()
    chart_bottom.type = "col"
    chart_bottom.title = "Bottom 10 Outlets — Net Sales Total (₹)"
    chart_bottom.style = 11
    chart_bottom.y_axis.title = "Net Sales Total"
    chart_bottom.x_axis.title = "Restaurant"
    data_ref2 = Reference(ws2, min_col=3, min_row=bottom_label_row + 1, max_row=bottom_data_end)
    cats_ref2 = Reference(ws2, min_col=2, min_row=bottom_data_start, max_row=bottom_data_end)
    chart_bottom.add_data(data_ref2, titles_from_data=True)
    chart_bottom.set_categories(cats_ref2)
    chart_bottom.width, chart_bottom.height = 20, 10
    ws2.add_chart(chart_bottom, "G21")

    # ---- Sheet 3: Raw Data ----
    raw_df = df.sort_values("Net Sales Total", ascending=False).reset_index(drop=True)
    ws3 = wb.create_sheet("Raw Data")
    raw_cols = list(raw_df.columns)
    write_header_row(ws3, 1, raw_cols)
    for r_idx, row in enumerate(raw_df.itertuples(index=False), start=2):
        for c_idx, val in enumerate(row, start=1):
            ws3.cell(row=r_idx, column=c_idx, value=val)
    ws3.freeze_panes = "A2"
    ws3.column_dimensions["A"].width = 32
    ws3.column_dimensions["B"].width = 16
    for i in range(3, len(raw_cols) + 1):
        ws3.column_dimensions[get_column_letter(i)].width = 14

    wb.save(output_xlsx)
    print("Saved:", output_xlsx)

In [2]:
generate_apsara_report(
    input_csv="apsara_sales_reports_Sales Report_Table (2).csv",
    output_xlsx="Apsara_Sales_Report_26November2026.xlsx",
    date_label="26th November 2026",
    report_label="Daily"
)

Saved: Apsara_Sales_Report_26November2026.xlsx
